In [1]:
%reload_ext autoreload
%autoreload 2

In [1]:
import os
import torch
import numpy as np
import random
import json
import argparse
import datetime
import wandb
from transformers import AutoModelForMaskedLM, Trainer, TrainingArguments, EarlyStoppingCallback
from preprocess_data import preprocess_dataset
from evals import generate_eval_table
from utils import printd

def set_seed(seed=42):
    """ Set all seeds to make results reproducible (deterministic mode).
        When seed is a false-y value or not supplied, disables deterministic mode. """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

from typing import List, Optional, Union, Any
# from dotenv import load_dotenv
# load_dotenv()


with open("config.json", "r") as file:
    config = json.load(file)

    

current_datetime = datetime.datetime.now()
formatted_datetime = current_datetime.strftime("%Y%m%d_%H-%M-%S")

model_output_dir = os.path.join(config.get("output").get("dir"), str(formatted_datetime))
os.makedirs(model_output_dir, exist_ok=True)
file = open(os.path.join(model_output_dir, config.get("output").get("log")), "w")

gpu_index_to_use = None
if gpu_index_to_use is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = gpu_index_to_use

printd("*" * 10 + "GPUs" + "*" * 10, file=file)
for i in range(torch.cuda.device_count()):
    printd(torch.cuda.get_device_properties(i).name, file=file)
printd("*" * 30, file=file)

# def plot_distribution(data):
#     # Create a seaborn style plot
#     sns.set(style="whitegrid")
    
#     # Create a figure and axis for the plot
#     plt.figure(figsize=(8, 6))
    
#     # Plot histogram and KDE (Kernel Density Estimate)
#     sns.histplot(data, kde=True, bins=30, color="blue", stat="density", linewidth=0)
    
#     # Set plot title and labels
#     plt.title("Distribution of the List", fontsize=16)
#     plt.xlabel("Value", fontsize=12)
#     plt.ylabel("Density", fontsize=12)
    
#     # Display the plot
#     plt.show()


/Users/sawale/Documents/NASA_IMPACT/indus_training/indus_training/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**********GPUs**********
******************************


In [4]:
# from datasets import load_dataset

# ds = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:100]")

# print(type(ds))

# from datasets import DatasetDict
# # ds = ds.train_test_split(test_size=0.2)
# val_split = 0.1
# test_split = 0.1

# if val_split is not None and test_split is not None:
#     temp_split = ds.train_test_split(test_size=(val_split + test_split), seed=42)
#     val_test_split = temp_split["test"].train_test_split(test_size=test_split / (val_split + test_split), seed=42)

#     ds = DatasetDict({
#         "train": temp_split["train"],
#         "validation": val_test_split["train"],
#         "test": val_test_split["test"],
#     })

# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("distilbert/distilroberta-base")
# # we are not concating because we want to chunk them letter
# def preprocess_function(examples):
#     return tokenizer(
#         examples["text"],
#         # truncation=True,
#         # padding="max_length",
#         # max_length=128,
#     )

# # the map can do the mapping seperate for different train and test set
# tokenized_ds = ds.map(
#     preprocess_function,
#     batched=True,
#     num_proc=4,
#     remove_columns=ds["train"].column_names,
# )

# def chunk_texts(examples, block_size=128):
#     results = {}
#     for k in examples.keys():
#         results[k] = []
#         # expecting k to be values like input_ids and attention_masks
#         for d in range(len(examples[k])):
#             # expecting d to be values for a document
#             if len(examples[k][d]) <= block_size:
#                 results[k].append(examples[k][d])
#             else:
#                 for i in range(0, len(examples[k][d]), block_size):
#                     results[k].append(examples[k][d][i:i + block_size])
        
#     return results 

# lm_dataset = tokenized_ds.map(chunk_texts, batched=True, num_proc=4)

# from transformers import DataCollatorForLanguageModeling

# tokenizer.pad_token = tokenizer.eos_token
# data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

In [5]:
from preprocess_data import preprocess_dataset

input_config = config.get("input")
lm_dataset, tokenizer, data_collator = preprocess_dataset(input_config)
model = AutoModelForMaskedLM.from_pretrained(config.get("input").get("model").get("hf"))


Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [6]:
training_args = TrainingArguments(
        output_dir=f"{config.get('output').get('model_backups_path')}timestamp_{formatted_datetime}/{config.get('input').get('model').get('hf')}/",
        **config.get("TrainingArguments")
    )
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=config.get("additional_training_config").get("training_patience")
)
trainer_config = dict(
        model=model,
        args=training_args,
        train_dataset=lm_dataset["train"],
        eval_dataset=lm_dataset["validation"],
        data_collator=data_collator,
        # compute_metrics=compute_metrics,
        callbacks=[early_stopping],
        tokenizer=tokenizer
    )
trainer = Trainer(**trainer_config)

/var/folders/p8/njq_1p954wl7whr0h7bm4gd00000gp/T/ipykernel_26414/409106401.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(**trainer_config)


In [7]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sajil (nasa-impact). Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
100,2.259600,2.116377
200,2.166800,2.033228


There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight', 'lm_head.decoder.bias'].


TrainOutput(global_step=265, training_loss=2.2114958565190155, metrics={'train_runtime': 109.6208, 'train_samples_per_second': 38.67, 'train_steps_per_second': 2.417, 'total_flos': 140545959535872.0, 'train_loss': 2.2114958565190155, 'epoch': 1.0})

In [8]:
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 7.64


In [46]:
eval_results

{'eval_loss': 2.800440549850464,
 'eval_model_preparation_time': 0.001,
 'eval_runtime': 53.9397,
 'eval_samples_per_second': 158.677,
 'eval_steps_per_second': 19.837}

In [29]:
model.save_pretrained("./test_model_final")
tokenizer.save_pretrained("./test_model_final")

('./test_model_final/tokenizer_config.json',
 './test_model_final/special_tokens_map.json',
 './test_model_final/vocab.json',
 './test_model_final/merges.txt',
 './test_model_final/added_tokens.json',
 './test_model_final/tokenizer.json')

In [70]:
# loading model from local

model = AutoModelForMaskedLM.from_pretrained("./test_model_final")

In [73]:
# inference

text = "The Milky Way is a <mask> <mask>."

In [74]:
from transformers import pipeline

mask_filler = pipeline("fill-mask", "./test_model_final")
mask_filler(text, top_k=3)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


[[{'score': 0.0767631009221077,
   'token': 22703,
   'token_str': ' galaxy',
   'sequence': '<s>The Milky Way is a galaxy<mask>.'},
  {'score': 0.05656205862760544,
   'token': 13258,
   'token_str': ' distant',
   'sequence': '<s>The Milky Way is a distant<mask>.'},
  {'score': 0.03904585912823677,
   'token': 44047,
   'token_str': ' galactic',
   'sequence': '<s>The Milky Way is a galactic<mask>.'}],
 [{'score': 0.6333543062210083,
   'token': 22703,
   'token_str': ' galaxy',
   'sequence': '<s>The Milky Way is a<mask> galaxy.'},
  {'score': 0.03247268497943878,
   'token': 9468,
   'token_str': ' universe',
   'sequence': '<s>The Milky Way is a<mask> universe.'},
  {'score': 0.02852061577141285,
   'token': 5518,
   'token_str': ' planet',
   'sequence': '<s>The Milky Way is a<mask> planet.'}]]

In [64]:
lm_dataset["test"]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 8559
})

In [24]:
for d, i in lm_dataset.items():
    print(i)

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 33200
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 8084
})


In [84]:
import random

# Randomly sample 100 examples from the test set
limited_test_dataset = lm_dataset["test"].select(random.sample(range(len(lm_dataset["test"])), 2))
test_results = trainer.predict(lm_dataset["test"])

RuntimeError: MPS backend out of memory (MPS allocated: 30.24 GB, other allocations: 8.66 MB, max allowed: 36.27 GB). Tried to allocate 6.14 GB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
test_results

In [67]:
from transformers import AutoTokenizer

# Load a tokenizer (replace 'bert-base-uncased' with your model's tokenizer)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Text to tokenize
text = "Hugging Face is great!"

# Tokenize with offset mappings
encoded = tokenizer(
    text,
    return_offsets_mapping=True,  # Include the offset mapping
)

# Tokens and their offsets
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
offsets = encoded["offset_mapping"]

# Print tokens and their positions
for token, (start, end) in zip(tokens, offsets):
    print(f"Token: {token}, Start: {start}, End: {end}, Substring: '{text[start:end]}'")


Token: [CLS], Start: 0, End: 0, Substring: ''
Token: hugging, Start: 0, End: 7, Substring: 'Hugging'
Token: face, Start: 8, End: 12, Substring: 'Face'
Token: is, Start: 13, End: 15, Substring: 'is'
Token: great, Start: 16, End: 21, Substring: 'great'
Token: !, Start: 21, End: 22, Substring: '!'
Token: [SEP], Start: 0, End: 0, Substring: ''


In [68]:
encoded

{'input_ids': [101, 17662, 2227, 2003, 2307, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1], 'offset_mapping': [(0, 0), (0, 7), (8, 12), (13, 15), (16, 21), (21, 22), (0, 0)]}

In [69]:
tokens

['[CLS]', 'hugging', 'face', 'is', 'great', '!', '[SEP]']

In [8]:
import pandas as pd

df = pd.read_json("../../data/sde_index_all.zip")

df.head()

,id,title,treepath,text
0,/SDE/interactive_multiinstrument_database_of_s...,Interactive Multi-Instrument Database of Solar...,/Heliophysics/Interactive Multi-Instrument Dat...,Interactive Multi-Instrument Database of Solar...
1,/SDE/interactive_multiinstrument_database_of_s...,About Interactive Multi-Instrument Database of...,/Heliophysics/Interactive Multi-Instrument Dat...,Interactive Multi-Instrument Database of Solar...
2,/SDE/interactive_multiinstrument_database_of_s...,Solar Flares Data Products,/Heliophysics/Interactive Multi-Instrument Dat...,\n\nAbout\n\nQuery Page\n\nData Sources\n\nDat...
3,/SDE/interactive_multiinstrument_database_of_s...,Solar Flares Database Help,/Heliophysics/Interactive Multi-Instrument Dat...,Solar Flares Database Help\n\nAbout\n\nQuery P...
4,/SDE/interactive_multiinstrument_database_of_s...,Solar Flares Data Sources,/Heliophysics/Interactive Multi-Instrument Dat...,Solar Flares Data Sources\n\nAbout\n\nQuery Pa...


In [15]:
df.shape

(190130, 4)

In [16]:
from datasets import load_dataset

# Load a CSV file
dataset = load_dataset("json", data_files="../../data/sde_index_all.zip")

dataset

/Users/sawale/Documents/NASA_IMPACT/indus_training/indus_training/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 190130 examples [00:04, 42230.41 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'treepath', 'text'],
        num_rows: 190130
    })
})

In [17]:
limited_dataset = dataset['train'].select(range(100))  # Select the first 100 rows


In [18]:
limited_dataset

Dataset({
    features: ['id', 'title', 'treepath', 'text'],
    num_rows: 100
})

In [11]:
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    PreTrainedTokenizer,
)


dataset_config = {
      "hf": {
        "path": "wikimedia/wikipedia",
        "name": "20231101.en",
        "split": "train[:100]"
      },
      "local": {
        "path": "json",
        "data_files": "../data/sde_index_all.zip"
      },
      "val_split": 0.1,
      "test_split": 0.1,
      "chunk_size": 128,
      "mlm_probability": 0.15,
      "text_column": "text"
    }
data_src = "hf"
DatasetDict(load_dataset(**dataset_config.get(data_src)))


ValueError: dictionary update sequence element #0 has length 4; 2 is required